In [1]:
# cell 1 — set up imports & paths
import sys
from pathlib import Path
import json
import pandas as pd

# -------------- 修改为你本地的项目根路径 --------------
PROJECT_ROOT = Path("/Users/shiyiliu/Desktop/Latent_Space_Vis").resolve()
# --------------------------------------------------------

# 插入 scripts/GNNGMMAnalyzer 到 sys.path（使 embedding_pipeline_core 可导入）
GNNGMM_DIR = PROJECT_ROOT / "scripts" / "GNNGMMAnalyzer"
if str(GNNGMM_DIR) not in sys.path:
    sys.path.insert(0, str(GNNGMM_DIR))

# 如果 backend 里有 util 也需要，可以加入 backend 目录
BACKEND_DIR = PROJECT_ROOT / "backend"
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("GNNGMM_DIR on sys.path:", GNNGMM_DIR.exists())
print("BACKEND_DIR on sys.path:", BACKEND_DIR.exists())

# 现在导入后端模块（和 backend/app.py 使用的同样函数）
from embedding_pipeline_core import load_embeddings_from_files
from embedding_pipeline import EmbeddingPipeline

print("Modules imported OK")

PROJECT_ROOT: /Users/shiyiliu/Desktop/Latent_Space_Vis
GNNGMM_DIR on sys.path: True
BACKEND_DIR on sys.path: True
Modules imported OK


In [2]:
# cell 2 — load embeddings and run pipeline
import torch   # 若 load_embeddings_from_files 内使用 torch
from pathlib import Path

# Embeddings 目录 —— 改为你本地路径（最好用绝对路径）
EMB_DIR = PROJECT_ROOT / "backend" / "anyembeddings"   # example: /Users/.../Latent_Space_Vis/anyembeddings

# 可选：如果你想指定具体 layer 文件与名字（与你 frontend 使用的一致）
layer_files = ["emb_unique_cas.pt","output0_unique_cas.pt","output1_unique_cas.pt"]
layer_names = ["emb_unique_cas","output0_unique_cas","output1_unique_cas"]
layer_files_dict = dict(zip(layer_names, layer_files))

print("Embeddings dir exists:", EMB_DIR.exists())
if not EMB_DIR.exists():
    raise FileNotFoundError(f"Embeddings dir not found: {EMB_DIR}")

# 调用 load_embeddings_from_files
# 如果你的 load_embeddings_from_files 接口支持只传目录也可以删掉第二个参数
embeddings = load_embeddings_from_files(EMB_DIR, layer_files_dict)
# embeddings 应该是 dict: { layer_name: numpy/tensor array (n_samples, dim) }
print("Loaded layers:", list(embeddings.keys()))
for k, v in embeddings.items():
    try:
        print(k, getattr(v, "shape", (len(v),)))
    except Exception:
        print(k, type(v))

# 构造 metadata（可选）
metadata = {"note": "notebook run"}

# 创建 pipeline（参考 backend/app.py 的用法）
tmp_output_dir = PROJECT_ROOT / "frontend" / "public" / "data" / "tmp_notebook"
tmp_output_dir.mkdir(parents=True, exist_ok=True)

pipeline = EmbeddingPipeline(
    embeddings=embeddings,
    output_dir=tmp_output_dir,
    id_mapping=None,    # 如果有 id map 的 json path，可加载并传入 dict
    n_components=4,     # 你想降维成多少维
    metadata=metadata
)

# 运行 GMM + MDS（和 backend 一样）
print("Fitting GMMs...")
pipeline.fit_gmms()
print("Computing MDS / distance embedding...")
pipeline.compute_mds()

# 生成 scatter + sankey DataFrame
scatter_df = pipeline.generate_scatter_plot_data()
sankey_df = pipeline.generate_sankey_data()

print("scatter rows:", len(scatter_df))
print("sankey rows:", len(sankey_df))

# 简单查看前几行
display(scatter_df.head())
display(sankey_df.head())

Embeddings dir exists: True
  ✓ emb_unique_cas: shape (152, 128)
  ✓ output0_unique_cas: shape (152, 256)
  ✓ output1_unique_cas: shape (152, 256)
Loaded layers: ['emb_unique_cas', 'output0_unique_cas', 'output1_unique_cas']
emb_unique_cas (152, 128)
output0_unique_cas (152, 256)
output1_unique_cas (152, 256)
Using 4 GMM components
Fitting GMMs...

Fitting GMMs for all layers...
  Fitting GMM for emb_unique_cas...
  Fitting GMM for output0_unique_cas...
  Fitting GMM for output1_unique_cas...
✓ Fitted GMMs for 3 layers
Computing MDS / distance embedding...

Computing MDS positions...
✓ Computed MDS positions for 3 layers

Generating scatter plot data...
✓ Generated scatter plot data: 456 points

Generating Sankey diagram data...
✓ Generated Sankey diagram data: 304 transitions
scatter rows: 456
sankey rows: 304


,id,x,y,color_value,layer
0,0,1053.377761,1234.786399,0.0,emb_unique_cas
1,1,1557.396823,921.892890,0.0,emb_unique_cas
2,2,910.397481,-2696.177409,0.0,emb_unique_cas
3,3,3084.250371,813.262800,0.0,emb_unique_cas
4,4,1441.918031,-237.268487,0.0,emb_unique_cas


,real_id,from_layer,to_layer,from_cluster,to_cluster
0,0,emb_unique_cas,output0_unique_cas,0,3
1,1,emb_unique_cas,output0_unique_cas,0,3
2,2,emb_unique_cas,output0_unique_cas,2,3
3,3,emb_unique_cas,output0_unique_cas,1,3
4,4,emb_unique_cas,output0_unique_cas,2,3


In [4]:
# cell 3 — save results to disk for later使用或前端读取
OUT_DIR = tmp_output_dir
scatter_csv = OUT_DIR / "scatter_notebook.csv"
sankey_csv = OUT_DIR / "sankey_notebook.csv"

scatter_df.to_csv(scatter_csv, index=False)
sankey_df.to_csv(sankey_csv, index=False)

# 如果你更喜欢 json（scatter 通常是 records）
scatter_json = OUT_DIR / "scatter_notebook.json"
sankey_json = OUT_DIR / "sankey_notebook.json"
scatter_df.to_json(scatter_json, orient="records")
sankey_df.to_json(sankey_json, orient="records")

print("Saved scatter ->", scatter_csv)
print("Saved sankey  ->", sankey_csv)

Saved scatter -> /Users/shiyiliu/Desktop/Latent_Space_Vis/frontend/public/data/tmp_notebook/scatter_notebook.csv
Saved sankey  -> /Users/shiyiliu/Desktop/Latent_Space_Vis/frontend/public/data/tmp_notebook/sankey_notebook.csv


In [20]:
from IPython.display import IFrame

IFrame(
    src="http://localhost:5173/?data=/tmp_notebook/sankey_notebook.json",
    width="100%",
    height=600
)